# Exercise 2.1: Loading Data & First Diagnostics (Angola IEA)

This notebook uses `IEA_2025_IV_TRIM_IND.sav`, the individual file of the
Inquerito ao Emprego em Angola (IEA), 4th quarter 2025, published by INE Angola.

It is a real SPSS export: 53,353 people, 206 columns, all variable and value
labels in Portuguese.

You will practice:
- Loading an SPSS file with `pd.read_spss()`
- Deciding whether to apply the file's value labels, and seeing what that costs
- Reading the codebook that ships inside the file
- Loading only the columns you need with `usecols`
- Running structural diagnostics with `info()`, `describe()` and `value_counts()`
- Spotting sentinel codes and empty columns before any cleaning

> **Pipeline:** this notebook only reads `0_raw/`. It writes nothing.

### Path Setup (run first)

> Use `os.path.join` for path construction.
> Required base path: `DATA_RAW_DIR = '../../data/0_raw/angola/employment_survey'`.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

DATA_RAW_DIR =   # your code here
RAW_FILE =   # your code here
raw_path = os.path.join(DATA_RAW_DIR, RAW_FILE)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Data path:', raw_path)
print('Exists?:', os.path.exists(raw_path))

---

## Task 1: Load the file and take a first look

`pd.read_spss()` reads SPSS `.sav` files. By default it applies the value labels
stored in the file, so coded variables come back as readable text.

In [ ]:
df_labelled =   # your code here

print('Shape:', df_labelled.shape)
df_labelled[['PROV', 'AREA_RESID', 'DEM_SEX', 'DEM_AGE']].head()

In [ ]:
df_labelled[['PROV', 'AREA_RESID', 'DEM_SEX']].  # your code here

In [ ]:
# Show a random sample of 5 rows (use random_state=0)
df_labelled[['PROV', 'AREA_RESID', 'DEM_SEX', 'DEM_AGE']].  # your code here

**Questions:**

- How many rows and columns does the file have? What is one row?
- Look at `PROV` and `DEM_SEX`. Are they text or numbers? Why?
- Is 206 columns a problem? For what?

---

## Task 2: The cost of applying value labels

Value labels are convenient, but they are applied to **every** labelled variable,
including ones where the label is a sentinel rather than a category. Reload with
`convert_categoricals=False` and compare the dtypes.

In [ ]:
df_raw = pd.read_spss(  # your code here: convert_categoricals=False )

comparison = pd.DataFrame({
    'labelled': df_labelled[['PROV', 'DEM_SEX', 'MJT_SYR', 'DEM_AGE']].dtypes,
    'raw_codes': df_raw[['PROV', 'DEM_SEX', 'MJT_SYR', 'DEM_AGE']].dtypes,
})
comparison

In [ ]:
# MJT_SYR is the year the person started their main job.
print('Labelled, first 5 values:')
print(df_labelled['MJT_SYR'].head().tolist())
print()
print('Raw codes, describe:')
print(df_raw['MJT_SYR'].describe())

**Questions:**

- What dtype does the labelled load give `MJT_SYR`? It is a year. Is that dtype
  usable for arithmetic?
- Run `describe()` on the raw `MJT_SYR`. What is the mean, and why is it not a
  plausible year?
- Which loading mode should the rest of this pipeline use, and why?

---

## Task 3: Read the codebook that ships inside the file

An SPSS file carries its own documentation: a label for every variable, and a
label for every coded value. The lesson shows this for Stata with
`variable_labels()`. For SPSS the equivalent lives in `pyreadstat`, and it can be
read **without loading the data**.

> This is the only cell in the whole series that imports `pyreadstat` directly.

In [ ]:
import pyreadstat

_, meta = pyreadstat.read_sav(  # your code here: metadataonly=True )

for name in ['PROV', 'AREA_RESID', 'DEM_SEX', 'DEM_AGE', 'WKT_USHRSTOT', 'MJT_SYR']:
    print(f'{name:15s} {meta.column_names_to_labels[name]}')

In [ ]:
# Value labels: the code to text mapping behind each categorical variable
label_set = meta.variable_to_label['PROV']
print('PROV has', len(meta.value_labels[label_set]), 'provinces')
print(meta.value_labels[label_set])

**Questions:**

- What does `WKT_USHRSTOT` actually measure? Could you have guessed from the name?
- How many provinces does `PROV` code, and what is the code range?
- Why read the metadata without loading the data?

---

## Task 4: Load only the columns you need with `usecols`

206 columns is more than this analysis needs. `usecols` tells the reader to skip
the rest entirely, so they never enter memory.

The 29 columns below carry the survey's demographic core, its labour module, the
interview date, and the survey weight.

In [ ]:
SPSS_COLS = [
    'NIDF', 'PPNO', 'G_06_ID_IEA', 'PROV', 'AREA_RESID', 'G_15_TRIMESTRE',
    'DEM_REL', 'DEM_SEX', 'DEM_AGE', 'DEM_MRT', 'DEM_EDL', 'S03_01',
    'ATW_PAY', 'ATW_PFT', 'ATW_FAM', 'ABS_JOB',
    'SRH_JOB', 'SRH_BUS', 'SRH_AVN', 'SRH_AVL', 'SRH_DES',
    'WKT_USHRSTOT', 'WKT_ACHRSTOT', 'MJT_SYR', 'MJJ_EMP_REL', 'GHVEDT',
    'POND_IEA_IV_TRIM_2025_IND', 'G_12', 'G_13',
]

df = pd.read_spss(  # your code here: usecols=SPSS_COLS, convert_categoricals=False )
print('Full file: ', df_raw.shape)
print('Subset:    ', df.shape)
df.head()

In [ ]:
full_mb = df_raw.memory_usage(deep=True).sum() / 1e6
subset_mb = df.memory_usage(deep=True).sum() / 1e6
print(f'Memory full:   {full_mb:8.2f} MB')
print(f'Memory subset: {subset_mb:8.2f} MB')
print(f'Saved:         {(1 - subset_mb / full_mb) * 100:8.1f}%')

**Questions:**

- How much memory did `usecols` save?
- What is the difference between `usecols` and loading everything then selecting?
- What happens if you misspell a name in `SPSS_COLS`? Try it.

---

## Task 5: Structural health check with `info()`

`df.info()` is the first diagnostic. Read the Non-Null Count column carefully:
it is where empty columns and skip patterns show up.

In [ ]:
df.  # your code here

In [ ]:
missing = pd.DataFrame({
    'n_missing': df.isna().sum(),
    'pct_missing': (df.isna().mean() * 100).round(1),
}).sort_values('pct_missing', ascending=False)
missing

**Questions:**

- Which two columns are entirely empty? What were they supposed to contain?
- The labour columns are around 78% missing. Is that damage, or something else?
  What decides the answer?
- `NIDF`, `PPNO` and `GHVEDT` are all `float64`. Is that right for any of them?

---

## Task 6: Summary statistics with `describe()`

`describe()` exposes impossible values. Look hard at every `min` and `max`.

In [ ]:
df[['DEM_AGE', 'WKT_USHRSTOT', 'WKT_ACHRSTOT', 'MJT_SYR', 'GHVEDT']].  # your code here

In [ ]:
df.describe(  # your code here: include='all' ).T

**Questions:**

- `MJT_SYR` has a mean of over 3000. What value is doing that, and how many are there?
- Find the same pattern in `WKT_USHRSTOT`. What is the sentinel?
- `DEM_AGE` runs 0 to 120. Which end is a real value and which is not?

---

## Task 7: Explore categories with `value_counts()`

`value_counts()` is the fastest way to see what is actually in a coded column.
Always pass `dropna=False` so the gaps are counted too.

In [ ]:
print(df['PROV'].  # your code here: value_counts(dropna=False).sort_index() )

In [ ]:
print(df['AREA_RESID'].value_counts(dropna=False))
print()
print(df['DEM_SEX'].value_counts(dropna=False))
print()
print(df['DEM_EDL'].value_counts(dropna=False).sort_index())

In [ ]:
print('Distinct households:', df['NIDF'].nunique())
print('Rows per household, describe:')
print(df['NIDF'].value_counts().describe())

**Questions:**

- How many province codes appear, and which province is largest?
- `DEM_EDL` codes go 1 to 7 and then 9, with no 8. Check the codebook: why?
- How many households are there, and how many people per household on average?

---

## Task 8: A quick visual sweep

Histograms of every numeric column at once are a fast way to spot sentinels: they
appear as a lonely spike far to the right of everything else.

In [ ]:
df[['DEM_AGE', 'WKT_USHRSTOT', 'WKT_ACHRSTOT', 'MJT_SYR',
    'DEM_EDL', 'POND_IEA_IV_TRIM_2025_IND']].hist(  # your code here: bins=30, figsize=(14, 8))
plt.tight_layout()
plt.show()

**Questions:**

- Two histograms show an isolated bar at the far right. Which, and what is it?
- What does the shape of the `DEM_AGE` distribution tell you about Angola?
- Is any of this data ready for analysis yet?